In [1]:
import sqlite3
import pandas as pd

In [2]:
# transaktsioonide andmebaas
transaction_db = "../example_data/v33_subset.db"

# loodud tabelid
vp_data_db = "../example_data/vp_data_actors.db"

# määruste tabelid
per_loc_db = "../example_data/db_per_loc.db"

# uus transaktsioonide tabel
transactions2 = "transaction_v2"

# tabelite nimed elusolendite ja kohtade jaoks
elustabel = 'elus_v1'
kohttabel = 'koht_v1'

In [3]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()

# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{transaction_db}" AS trans')

# määruste andmebaasi lisamine (maarused)
cur.execute(f'ATTACH DATABASE "{per_loc_db}" AS maarused')

## Create new transaction table

### juurde lisada veerud koht ja elus, kus on listide põhjal otsus sõna kohta

In [4]:
%%time
cur.execute("""DROP TABLE if exists {new_table}""".format(new_table = transactions2))
cur.execute("""CREATE TABLE {new_table} as SELECT * FROM trans.'transaction' """.format(new_table = transactions2))

CPU times: user 3.49 ms, sys: 2.31 ms, total: 5.8 ms
Wall time: 13.6 ms


In [5]:
# add default values for koht and elus
cur.execute("""
ALTER TABLE {new_table}
ADD koht VARCHAR(50) DEFAULT 'UNK'
""".format(new_table = transactions2))
con.commit()

cur.execute("""
ALTER TABLE {new_table}
ADD elus VARCHAR(50) DEFAULT 'UNK'
""".format(new_table = transactions2))
con.commit()

#### fill in new columns

In [7]:
%%time

cur.execute("""
UPDATE {new_table}
SET elus = 'YES'
WHERE lower({new_table2}.lemma) in (select lower(lemma) from maarused.{elustbl})
""".format(new_table = transactions2, new_table2 = transactions2, elustbl=elustabel))

con.commit()

CPU times: user 7.36 ms, sys: 705 µs, total: 8.06 ms
Wall time: 11.8 ms


In [8]:
%%time

cur.execute("""
UPDATE {new_table}
SET koht = 'YES'
WHERE lower({new_table2}.lemma) in (select lower(lemma) from maarused.{kohttbl})
""".format(new_table = transactions2, new_table2 = transactions2, kohttbl=kohttabel))

con.commit()

CPU times: user 3.03 ms, sys: 2.5 ms, total: 5.52 ms
Wall time: 10.6 ms


## kontroll

In [10]:
query = """SELECT * from {new_table} 
where koht='YES'
limit 5""".format(new_table = transactions2)
source = pd.read_sql_query(query, con)
source

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,koht,elus
0,215,152,19,-3,nsubj,uks,uks,"com,nom,sg",None,S,YES,UNK
1,249,172,6,2,obj,eriala,eriala,"com,nom,sg",None,S,YES,UNK
2,428,274,7,-2,obl,kohvikus,kohvik,"com,in,sg",None,S,YES,UNK
3,458,295,3,-1,obl,Lennujaamas,lennujaam,"com,in,sg",None,S,YES,UNK
4,463,296,11,1,obl,liiklusummikusse,liiklusummik,"com,ill,sg",None,S,YES,UNK


In [11]:
con.close()